# SSP/VSA embeddings in MiniGrid
There are wrappers and features extractors included in this package that are are for the MiniGrid environments specfically.


In the grid world environments, each cell can contain, at most, one object, which is specified by its type, colour, and state. Possible object types include wall, door, key, ball, box, goal, and lava,
with each object having attributes like colour (from a predefined set) and states (open, closed, locked) that are specific to certain object types.

The agent has a limited $7\times7$ field of view and cannot see through walls. The default observations are represented as a $7\times7\times3$ integer matrix, where each vector $(i,j,:)$ denotes the type, colour, and state of the object at position $(i,j)$ within the agent's field of view.  The agent can perform seven actions: turn left, turn right, move forward, pick up an object, drop an object, open a door or box, and complete a task (which is not applicable in the tasks considered here).

## Wrappers
- **SSPMiniGridPoseWrapper:** Represents the agent's pose within the environment as an SSP,
\begin{align}
   \phi_{\text{pose}} = \phi \left ( \left [x,y,\theta \right ] \right ) 
\end{align}
where $x,y$ is the agent's global position in the grid and $\theta \in \{0,1,2,3\}$ is an integer indicating the direction the agent is facing. Although the state variables are discrete due to the finite number of possible agent positions and orientations, they are treated as continuous variables in this embedding.
- **SSPMiniGridViewWrapper:**  Uses the algebra of HRRs to encode both the agent's field of view and its pose. The information encoded includes a representation of the agent's pose (global position and orientation), $\phi([x,y,\theta])$; a representation of the object the agent is carrying (bound with a semantic pointer, $\mathtt{HAS}$), if the agent is carrying an object (in these environments the agent is limited to carrying a single object, so the sum over objects carried in in equations below is over at most a single object); and a bundled representation of objects in the agent's field of view and their location relative to the agent. There are two versions of this:
    - **obj_encoding='allbound':** The complete state encoding is constructed via binding and bundling operations:
\begin{align}
   \Phi_{\text{view}} = \phi([x,y,\theta]) + \mathtt{HAS} \, \circledast &\sum_{\text{objects carried}}  \mathtt{ITEM}_i \circledast \mathtt{COLOUR}_i \circledast \mathtt{STATE}_i  \\
     + &\sum_{\text{objects in view}} \Delta\phi_i \circledast \mathtt{ITEM}_i \circledast \mathtt{COLOUR}_i \circledast \mathtt{STATE}_i. 
\end{align}
The vector, $\mathtt{ITEM}$, indicates the 'type' of an object in view, and can take on values $\mathtt{DOOR}$, $ \mathtt{KEY}$, $ \mathtt{BALL}$, $ \mathtt{BOX}$, $ \mathtt{GOAL}$, or $ \mathtt{LAVA}$. The vector, $\mathtt{COLOUR}$, indicates the colour of the associated object. The vector, $\mathtt{STATE}$, indicates the 'state' of an object, and can take on values $\mathtt{OPEN}$, $ \mathtt{CLOSED}$, or $ \mathtt{LOCKED}$ (objects with fixed states, such as lava or balls, are encoded as being in the `open' state). Finally, $\Delta\phi_i$, encodes an object-in-view's location relative to the agent.
    - **obj_encoding='slotfiller':** The complete state encoding is constructed via binding and bundling operations in a slot-filler style:
\begin{align}
    \Phi_{\text{slot-filler}} = \phi([x,y,\theta]) + \mathtt{HAS} \, \circledast &\sum_{\text{objects carried}} \left ( \mathtt{ITEM} \circledast \mathtt{I}_i + \mathtt{COLOUR} \circledast \mathtt{C}_i + \mathtt{STATE} \circledast \mathtt{S}_i \right )\\
     + &\sum_{\text{objects in view}} \Delta\phi_i \circledast \left ( \mathtt{ITEM} \circledast \mathtt{I}_{i} + \mathtt{COLOUR} \circledast \mathtt{C}_i + \mathtt{STATE} \circledast \mathtt{S}_i \right ),   
\end{align}
where $\mathtt{ITEM}$, $\mathtt{COLOUR}$, and $\mathtt{STATE}$ are random vectors that represent **slots** -- they indicate the type of the vector they are bound with -- while $\mathtt{I}_i$, $\mathtt{C}_i$, and $\mathtt{S}_i$ denote the actual **values** of item type, colour, and state. The main difference between $\Phi_{\text{slot-filler}}$ and the prior \gls*{hrr} embedding, $\Phi_{\text{view}}$, is representational overlap.
In $\Phi_{\text{view}}$,  objects differing in any attribute (\eg an open blue door versus a closed blue door) are dissimilar, whereas in $\Phi_{\text{slot-filler}}$, objects sharing properties have greater similarity (e.g., the representation of an open blue door is more similar to a closed blue door or a blue key compared to a red box).
    - **Local vs gloabl:** (view_type='local' or 'global') In local mode we use  $\Delta\phi_i$, object-in-view's location relative to the agent. While in global mode, we  $\phi_i$ instead, an object-in-view's global location in the env
- **SSPMiniGridMissionWrapper:** Added on to the above encoding is a representation of the mission string -- a part of the observation space in some MiniGrid and all BabyAI tasks.
    - Examples of misssion statements: “go to the {color} door”, “pick up the {color} {type}”, “go to a/the {color} {type}” + “and go to a/the {color} {type}” + “, then go to a/the {color} {type}” + “and go to a/the {color} {type}”
    - This class is a work-in-progress. Currently, regex is used to decompose the string, looking for particular command patterns (e.g., "go to _", "fetch a _", "pick up a _", "open the _", "put the _ near the _") as well as object and color names. The idea is to break up the mission statement into different simple subcommands that each involve a sngle object and binding a command type representations (e.g., $\mathtt{GO\_TO}$, $\mathtt{PICK\_UP}$, $\mathtt{OPEN}$) to object color and type representations (those used in the view encoding). This class will likely change in future versions of this package.
- **SSPMiniGridWrapper** An interface to selct one of the above. Takes input encode_pose (true/false),encode_view (true/false), encode_mission (true/false). Currently encode_mission=True with encode_view=False is not supported.

In [1]:
import json
import torch
import numpy as np
import random
import os
import pandas as pd
import sys
import random
import math
import argparse
import yaml
from pathlib import Path
import datetime
import wandb
import pickle
from tqdm import tqdm
from functools import reduce
import gymnasium as gym, contextlib, io

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download

import plotly.graph_objects as go

from typing import List, Optional
import fire

import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from typing import List

from fairscale.nn.model_parallel.initialize import (
    get_model_parallel_rank,
    initialize_model_parallel,
    model_parallel_is_initialized,
)

######################################################

# Parser
curr_date = datetime.datetime.now().strftime("%Y%m%d")

def str2bool(v):
    if isinstance(v, bool):
        return v
    if v.lower() in ("yes", "true", "t", "1"):
        return True
    elif v.lower() in ("no", "false", "f", "0"):
        return False
    else:
        raise argparse.ArgumentTypeError("Boolean value expected.")

/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import gymnasium as gym
import minigrid
from stable_baselines3 import PPO

import sys, os
sys.path.insert(1, os.path.dirname(os.getcwd()))
os.chdir("..")
import vsagym
from vsagym.wrappers import minigrid_wrappers

env = gym.make('MiniGrid-Empty-5x5-v0')
env = minigrid_wrappers.SSPMiniGridPoseWrapper(env,
                             shape_out=251,
                             decoder_method='from-set')
observation, _ = env.reset()
for t in range(5):
    action = env.action_space.sample()
    observation, _, terminated, truncated, _ = env.step(action)
    if terminated or truncated or t == 4:
        observation, _ = env.reset()
env.close()

env = gym.make('MiniGrid-KeyCorridorS3R1-v0')
env = minigrid_wrappers.SSPMiniGridViewWrapper(env,
                                               obj_encoding='allbound',
                                               view_type='local',
                                               shape_out=1024,
                                               decoder_method='from-set')
observation, _ = env.reset()


/home/vdhanraj/vsa-gym-wrapper/vsagym/spaces/ssp_box.py:107: UserWarning: Box bound precision lowered by casting to float32
  warnings.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: uint8
  logger.warn(
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarnin

In [3]:
curr_date = datetime.datetime.now().strftime("%Y%m%d")

# === Load config file (from train_encoders_and_decoders.py default values) ===
with open("Neurosymbolic-LLM/Programs/train_encoders_and_decoders_default_config.yaml", "r") as f:
    config_defaults = yaml.safe_load(f)

# === Set all variables from config as globals ===
for k, v in config_defaults.items():
    globals()[k] = v

# === Load config file (from fine_tune_decoders.py default values) ===
with open("Neurosymbolic-LLM/Programs/fine_tune_decoders_default_config.yaml", "r") as f:
    fine_tune_decoder_config_defaults = yaml.safe_load(f)

# Set all variables directly from config
for k, v in fine_tune_decoder_config_defaults.items():
    globals()[k] = v

# === Explicit overrides from code ===
master_port = "25000"
run_name = "1B_param_model"

curr_dir = '~/vsa-gym-wrapper/Neurosymbolic-LLM/Programs/'
git_dir = '~/vsa-gym-wrapper/Neurosymbolic-LLM'
ckpt_dir = '~/.llama/checkpoints/Llama3.2-1B-Instruct/original'
tokenizer_path = '~/.llama/checkpoints/Llama3.2-1B-Instruct/original/tokenizer.model'
#encoder_path = "/home/vdhanraj/test/Neurosymbolic-LLM/Programs/models/encoders_1B_param_model.pth"
#decoder_path = "/home/vdhanraj/test/Neurosymbolic-LLM/Programs/models/decoders_1B_param_model.pth"

max_batch_size = 1
n_samples = max_batch_size
train_data_rounds = 10000 // n_samples
val_data_rounds   = 100   // n_samples
test_data_rounds  = 1000  // n_samples
# train_data_rounds = 1000 // n_samples
# val_data_rounds   = 10   // n_samples
# test_data_rounds  = 100  // n_samples

symbolic_encoding_layer = 12
symbolic_decoding_layers = [12]
layer_numbers = range(0,17)
VSA_dim = observation['image'].shape[0]
model_dim = 2048
log_wandb = True
run_name = "Local Test"

# === Path expansion for file/dir fields ===
curr_dir              = str(Path(curr_dir).expanduser())
git_dir               = str(Path(git_dir ).expanduser())
ckpt_dir              = str(Path(ckpt_dir).expanduser())
tokenizer_path        = str(Path(tokenizer_path).expanduser())
training_data_df_path = str(Path(training_data_df_path).expanduser()) if training_data_df_path else ''
val_data_df_path      = str(Path(val_data_df_path).expanduser()) if val_data_df_path else ''
testing_data_df_path  = str(Path(testing_data_df_path).expanduser()) if testing_data_df_path else ''

layer_numbers = torch.tensor(layer_numbers)
n_samples         = min(n_samples,         max_batch_size)
val_n_samples     = min(val_n_samples,     max_batch_size)
testing_n_samples = min(testing_n_samples, max_batch_size)

print("Configuration loaded successfully")
print(f"Run name: {run_name}")
print(f"Master port: {master_port}")
print(f"Model dimension: {model_dim}")
print(f"VSA dimension: {VSA_dim}")
print(f"Log to W&B: {log_wandb}")

sys.path.insert(0, git_dir)

from llama.encoder_decoder_networks import Encoder, Decoder, Encoder_Deep, Decoder_Deep, LastTokenTransformer
from llama.vsa_engine import *
from llama.utilities import *
from llama import Llama

print("LLaMA modules imported successfully")


Configuration loaded successfully
Run name: Local Test
Master port: 25000
Model dimension: 2048
VSA dimension: 969
Log to W&B: True
LLaMA modules imported successfully


In [4]:
if log_wandb:
    wandb.finish() # If there is an active current run, terminate it
    wandb.init(
        project = "Symbolic LLM Demo",
        name    = run_name,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", torch.cuda.get_device_name(torch.cuda.current_device()))

os.environ['RANK'] = "0"
os.environ['WORLD_SIZE'] = "1"
os.environ['MASTER_ADDR'] = "127.0.0.2"
os.environ['MASTER_PORT'] = master_port
os.environ['LOCAL_RANK']  = "0"
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

generator = Llama.build(
    ckpt_dir=ckpt_dir,
    tokenizer_path=tokenizer_path,
    max_seq_len=max_seq_len,
    max_batch_size=max_batch_size,
)

# Freeze the pretrained LLM
for param in generator.model.parameters():
    param.requires_grad = False


wandb: Currently logged in as: varun_dhanraj to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Current device: NVIDIA GeForce RTX 4090
> initializing model parallel with size 1
> initializing ddp with size 1
> initializing pipeline with size 1
Loaded in 1.65 seconds


In [5]:
def generate_dialogs(missions):
    dialogs = []
    for batch in range(len(missions)):
        curr_problem = missions[batch]
        dialog = [
            {
                "role": "system",
                "content":
                (
                    "You are an agent exploring an environment, guided by a mission statement. Follow the format of previous answers and follow the following instructions. "
                    "Your moves are an agent are as follows:\n"
                    "– You can change the direction you are facing, but outputting *left* or *right*\n"
                    "– You can move forward by outputting *forward*\n"
                    "– You can take and drop an object by outputting *take* or *drop*\n"
                    "– You can toggle the state of the object in front of you by ouptutting *toggle*\n"
                    "- You can take no action by outputting *done*"
                )
            },
            {"role": "user", "content": "Your mission is: go to the red ball."},
            {"role": "assistant", "content": "forward"},
            {"role": "assistant", "content": "left"},
            {"role": "assistant", "content": "forward"},
            {"role": "assistant", "content": "forward"},
            {"role": "assistant", "content": "right"},
            {"role": "assistant", "content": "forward"},
            {"role": "user", "content": "Your mission is: open the blue door."},
            {"role": "assistant", "content": "right"},
            {"role": "assistant", "content": "right"},
            {"role": "assistant", "content": "forward"},
            {"role": "assistant", "content": "forward"},
            {"role": "assistant", "content": "forward"},
            {"role": "assistant", "content": "toggle"},
            {"role": "user", "content": f"Your mission is: {curr_problem}"}
        ]
        dialogs += [dialog]
    return dialogs

In [6]:
minigrid_actions = {
    "left":    np.int64(0),
    "right":   np.int64(1),
    "forward": np.int64(2),
    "pickup":  np.int64(3),
    "drop":    np.int64(4),
    "toggle":  np.int64(5),
    "done":    np.int64(6),
}


In [7]:
symbolic_decoding_layers = [10]

In [8]:
generator.model.symbolic_decoding_layers  = symbolic_decoding_layers
decoders = nn.ModuleList()
encoders = nn.ModuleList()
for n_layer in layer_numbers:
    decoder = Decoder(n_layer, VSA_dim, generator.model.output.weight.shape[1]).to(device)
    decoders.append(decoder)
generator.model.decoders = decoders

for n_layer in layer_numbers:
    encoder = Encoder(n_layer, generator.model.output.weight.shape[1], VSA_dim).to(device)
    encoders.append(encoder)
generator.model.encoders = encoders

if rms_layer:
    generator.model.rms_layers = [] 
    for sl in symbolic_decoding_layers:
        if sl != 17:
            generator.model.rms_layers.append(RMSNorm(generator.model.output.weight.shape[1], eps=1e-05)) # params.dim, eps=params.norm_eps
        else:
            generator.model.rms_layers.append(RMSNorm(generator.model.output.weight.shape[0], eps=1e-05)) # num_tokens, eps=params.norm_eps
else:
    generator.model.skip_weights = nn.Parameter(torch.zeros(len(symbolic_decoding_layers)) + starting_skip_strength)

for param in generator.model.parameters():
    param.requires_grad = False
for sl in symbolic_decoding_layers:
    for param in generator.model.decoders[sl].parameters():
        param.requires_grad = True
for sl in symbolic_decoding_layers:
    for param in generator.model.encoders[sl].parameters():
        param.requires_grad = True
if rms_layer:
    for r_layer in generator.model.rms_layers:
        for param in r_layer.parameters():
            param.requires_grad = True

# Delete unnecessary layers to save memory
for i in range(len(generator.model.decoders)):
    if i not in symbolic_decoding_layers:
        del generator.model.decoders[i]  # Delete layer
        generator.model.decoders.insert(i, None)  # Insert None to maintain indexing

for i in range(len(generator.model.encoders)):
    if i not in  symbolic_decoding_layers:
        del generator.model.encoders[i]  # Delete layer
        generator.model.encoders.insert(i, None)  # Insert None to maintain indexing


In [9]:
def llm_get_action(observations, generator, temperature=0.7, reference_tokens=None, max_decoding_length=5,
                   inference_mode=None, verbose=False):
    missions = []
    states   = []
    for obs in observations:
        missions += [obs['mission']]
        states   += [obs['image']]
    states = torch.tensor(states)
    dialogs = generate_dialogs(missions)
    h_stacks, list_of_probs, list_of_logits, out_tokens = episode(dialogs=dialogs, generator=generator,
                                                                  inference_mode=inference_mode,
                                                                  curr_symbol=states, temperature=temperature,
                                                                  reference_tokens=reference_tokens,
                                                                  max_decoding_length=max_decoding_length, verbose=verbose
                                                                 )
    responses = [generator.tokenizer.decode(out_tokens[i]) for i in range(len(out_tokens))]
    actions = []
    for r in responses:
        if r in minigrid_actions:
            if verbose:
                print("Chosen Action:", r)
            actions += [minigrid_actions[r]]
        elif r[1:-1] in minigrid_actions:
            if verbose:
                print("Chosen Action:", r)
            actions += [minigrid_actions[r[1:-1]]]
        else:
            if verbose:
                print("INVALID RESPONES:", r)
            actions += [6]
    return actions, out_tokens, list_of_probs


In [10]:
from minigrid.wrappers import RGBImgObsWrapper, ImgObsWrapper

# Todo: Make this run in batch (i.e., LLM multibatch). Anytime there is an index on 0, it's probably something to fix
def run_RL_episode(env_name, generator, temperature=0.7, reference_tokens=None, seed=0, max_steps=100, 
                   max_decoding_length=5, inference_mode=None, verbose=False):
    # Reference tokens is either None (in which case, do normal inference) or its a list of length batch_length, 
    #  with each element being of length of the trajectory which generated it. Each element of the trajectory will be 
    #  a list of token_ids of length response per action (usually will be 1 token long)
    # Setup environment
    env = gym.make(env_name)
    env = minigrid_wrappers.SSPMiniGridPoseWrapper(env, shape_out=VSA_dim, decoder_method='from-set')

    # Initialize
    observation, _ = env.reset(seed=seed)
    total_reward = 0
    step_count = 0
    max_steps = max_steps  # optional safety limit

    terminated = False
    truncated  = False
    actions       = [[]] # The actions that the LLM agent took
    trajectories  = [[]] # The list of tokens that describe the LLMs actions
    probabilities = [[]] # The probabilities of all tokens for each token predicted by the LLM

    while not terminated and not truncated and step_count < max_steps:
        # Get action from LLM agent
        observations = [observation]

        if verbose and step_count == 0:
            print("mission:", observations[0]["mission"])

        curr_ref_tokens = None if reference_tokens is None else reference_tokens[0][step_count]
        acts, trajs, probs = llm_get_action(observations=observations, generator=generator, temperature=temperature,
                                            reference_tokens=curr_ref_tokens, max_decoding_length=max_decoding_length,
                                            inference_mode=inference_mode, verbose=verbose)

        actions      [0] += [acts [0]]
        trajectories [0] += [trajs[0]]
        probabilities[0] += [probs[:,0]]

        # Take step
        observation, reward, terminated, truncated, _ = env.step(acts[0])
        total_reward += reward
        step_count += 1

        if verbose:
            print(f"Step {step_count}: action={acts[0]}, reward={reward}, total_reward={total_reward}")

    total_reward = [total_reward]
    # End of episode
    env.close()
    if verbose:
        print(f"Episode completed in {step_count} steps with total reward: {total_reward[0]}")
    return total_reward, trajectories, probabilities

In [11]:
params = list(filter(lambda p: p.requires_grad, generator.model.parameters()))
print("Number of trainable parameters:", sum(p.numel() for p in params))


Number of trainable parameters: 3969024


In [12]:
print(torch.cuda.memory_allocated() / 1e6, "MB allocated")
print(torch.cuda.memory_reserved()  / 1e6, "MB reserved")


3356.774912 MB allocated
3504.340992 MB reserved


In [13]:
#reward, trajectory, probabilities = run_RL_episode(env_name='MiniGrid-Empty-5x5-v0', generator=generator, verbose=False)

In [14]:
lr             = 1e-5               # start here for 7-30 B models
betas          = (0.9, 0.95)         # HF-TRL defaults, good with PPO/GRPO
weight_decay   = 0.01
eps            = 1e-8                # keep default unless you hit NaNs
max_grad_clip  = 0.1                 # clip *after* .backward(), before .step()
beta = 0.25
lmbda = 0.0


In [15]:
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, generator.model.parameters()), lr=lr, 
                              #betas=betas, weight_decay=weight_decay, eps=eps
                             )


In [ ]:
n_trials = 1 # Number of runs to gather before starting training process
n_groups = 10
minibatch_size = 10
n_epochs = 100
max_steps = 50
max_decoding_length = 5
grpo_training_loop = 100
verbose = True

temperature = 0.7
#inference_mode = generator.model.forward_rl
inference_mode = generator.model.forward_rl_enc_dec

losses = []
total_rewards = []
for curr_epoch in range(n_epochs):
    batch_dialogs = []
    trajectories  = []
    model_probs   = []
    advantages    = []

    env_name = 'MiniGrid-Empty-5x5-v0'

    print(f"Epoch number {curr_epoch}", flush=True)

    generator.model.eval()
    for trial in range(n_trials):
        if verbose:
            print(f" - Trial number {trial}", flush=True)
        group_rewards = []
        for group_run in range(n_groups):
            if verbose: 
                print(f" ----- Group run {group_run}", flush=True)
            with torch.no_grad():
                curr_seed = group_run + n_groups * trial
                rewards, trajs, probs = run_RL_episode(env_name=env_name, generator=generator, temperature=temperature,
                                                       reference_tokens=None, seed=curr_seed, max_steps=max_steps, 
                                                       max_decoding_length=max_decoding_length,
                                                       inference_mode=inference_mode, verbose=False)

                # trajectory and probabilities are lists of length batch_size, where each element 
                #  is a list of length number_of_moves. For trajectory, each element is a list of integers, 
                #  which represent the tokens outputted by the model for that guess. For probabilities, 
                #  each element is a tensor of shape (num_output_tokens, num_possible_tokens), where 
                #  for llama num_possible_tokens = 128256

            selected_probs = []
            for batch in range(len(probs)):
                for move_number in range(len(probs[batch])):
                    selected_probs += [probs[batch][move_number].gather(
                        1, torch.tensor(trajs[batch][move_number]).unsqueeze(1)).squeeze(1)]

            batch_dialogs += [(env_name, curr_seed)]
            trajectories  += [trajs]
            model_probs   += [selected_probs]

            group_reward = [rewards[i] for i in range(len(rewards))]
            group_rewards += group_reward
            total_rewards += group_reward
        ave_reward = np.mean(group_rewards)
        std_reward = np.std(group_rewards)
        adv_per_group = (group_rewards - ave_reward) / (std_reward + 1e-2)

        advantages += list(adv_per_group)

    print("Average Reward:", np.mean(total_rewards[-n_groups*n_trials:]), flush=True)
    if log_wandb:
        wandb.log({"average_reward": np.mean(total_rewards[-n_groups*n_trials:])})

    generator.model.train()
    optimizer.zero_grad()

    backprop_count = 0
    running_loss_log = []
    clip_fractions = []
    if len(losses) > 10000:
        losses = losses[-5000:]

    for loss_run in range(grpo_training_loop):
        n = np.random.randint(low=0, high=len(trajectories))
        # ------------------------------------------------------------
        # 1) Re‑evaluate trajectory with current policy
        # ------------------------------------------------------------
        _, _, pi_new_probs = run_RL_episode(
            env_name=batch_dialogs[n][0],
            generator=generator,
            temperature=temperature,
            reference_tokens=trajectories[n],
            seed=batch_dialogs[n][1],
            max_steps=max_steps,
            inference_mode=inference_mode,
            verbose=False,
        )

        # ------------------------------------------------------------
        # 2) Gather token probabilities along the actions we actually took
        # ------------------------------------------------------------
        pi_new_probs_filtered = []               # list(bsz) ● each is list(tokens) ● each element is 1‑D tensor
        for b in range(len(pi_new_probs)):       # b = batch index (within this trajectory)
            token_probs = []
            for move_no in range(len(pi_new_probs[b])):
                gathered = pi_new_probs[b][move_no].gather(
                    1, torch.tensor(trajectories[n][b][move_no],
                                     device=pi_new_probs[b][move_no].device).unsqueeze(1)
                ).squeeze(1)                     # shape (n_tokens_in_move,)
                token_probs.append(gathered)
            pi_new_probs_filtered.append(token_probs)

        # pi_old_probs_filtered was stored earlier on GPU; detach & move to CPU now
        pi_old_probs_filtered = [seq.detach() for seq in model_probs[n]]

        # ------------------------------------------------------------
        # 3) Compute PPO / GRPO ratios and losses
        # ------------------------------------------------------------
        pi_new_probs_filtered = [torch.clamp(seq, min=1e-8) for seq in pi_new_probs_filtered[0]]
        pi_old_probs_filtered = [torch.clamp(seq, min=1e-8) for seq in pi_old_probs_filtered]

        log_new_probs = [torch.log(seq) for seq in pi_new_probs_filtered]
        log_old_probs = [torch.log(seq) for seq in pi_old_probs_filtered]   # ← fixed typo

        r = [torch.exp(log_new_probs[i] - log_old_probs[i])
             for i in range(len(log_old_probs))]
        clamped_r = [torch.clamp(r[i], 1 - max_grad_clip, 1 + max_grad_clip)
                     for i in range(len(r))]

        clip_fraction = ([r[i] != clamped_r[i] for i in range(len(r))])
        clip_fractions += [torch.concat(clip_fraction).to(float).detach().mean()]

        # advantage for this trajectory
        adv = advantages[n]

        grpo_loss_tok = [torch.min(r[i] * adv, clamped_r[i] * adv)
                         for i in range(len(r))]
        grpo_loss = torch.cat(grpo_loss_tok).mean()     # scalar

        # entropy (exclude last token’s <eos>)
        entropy_tok = [-(pi_new_probs[0][i][:-1] *
                         torch.log(pi_new_probs[0][i][:-1] + 1e-8)).sum(-1)
                       for i in range(len(pi_new_probs[0]))]
        entropy_loss = torch.cat(entropy_tok).mean()

        # ------------------------------------------------------------
        # 4) Scale, back‑prop, and free memory immediately
        # ------------------------------------------------------------
        scaled_loss = -(grpo_loss + lmbda * entropy_loss) / minibatch_size
        #scaled_loss = (grpo_loss + lmbda * entropy_loss) / minibatch_size
        losses += [scaled_loss.item()]
        scaled_loss.backward()                       # only one example’s graph is live
        running_loss_log.append(scaled_loss.detach().item())

        # explicit cleanup
        del pi_new_probs, pi_new_probs_filtered, log_new_probs, log_old_probs, r, clamped_r
        torch.cuda.empty_cache()                     # optional, shows drop in nvidia‑smi

        # ------------------------------------------------------------
        # 5) Optimizer step every `minibatch_size` examples
        # ------------------------------------------------------------
        if (loss_run + 1) % minibatch_size == 0:
            optimizer.step()
            optimizer.zero_grad()
            backprop_count += 1
            if verbose:
                print(f"Minibatch {(loss_run + 1) // minibatch_size} Loss: {np.mean(running_loss_log):.4f}" + f", Clipped fraction: {torch.stack(clip_fractions).mean().item()}",
                      flush=True)
            running_loss_log  = []
            clipped_fractions = []

    # (optionally) log mean loss for the epoch
    if verbose and running_loss_log:
        print("Average Loss this epoch:",
              np.mean(running_loss_log[-backprop_count * minibatch_size:]), flush=True)
    if log_wandb:
        wandb.log({"average_loss": np.mean(running_loss_log[-backprop_count * minibatch_size:])})

    

Epoch number 0
 - Trial number 0
 ----- Group run 0


/home/vdhanraj/vsa-gym-wrapper/vsagym/spaces/ssp_box.py:107: UserWarning: Box bound precision lowered by casting to float32
  warnings.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: uint8
  logger.warn(
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarnin

 ----- Group run 1
 ----- Group run 2
 ----- Group run 3
 ----- Group run 4
 ----- Group run 5
 ----- Group run 6
 ----- Group run 7
 ----- Group run 8
 ----- Group run 9
Average Reward: 0.3956
Minibatch 1 Loss: 0.0272, Clipped fraction: 0.4383770178226842
Minibatch 2 Loss: -0.0158, Clipped fraction: 0.38425446460791646
Minibatch 3 Loss: 0.0174, Clipped fraction: 0.37245097709918
Minibatch 4 Loss: -0.0616, Clipped fraction: 0.39057831218946437
Minibatch 5 Loss: 0.0249, Clipped fraction: 0.397679142254979
Minibatch 6 Loss: -0.0175, Clipped fraction: 0.40721812913813793
Minibatch 7 Loss: 0.0291, Clipped fraction: 0.4030625563951909
Minibatch 8 Loss: -0.0004, Clipped fraction: 0.40621547803315394
Minibatch 9 Loss: -0.0597, Clipped fraction: 0.40546151653629275
Minibatch 10 Loss: 0.0209, Clipped fraction: 0.40605904900676093
Epoch number 1
 - Trial number 0
 ----- Group run 0


/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/vdhanraj/anaconda3/envs/minigrid/lib/python3.11/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


 ----- Group run 1
 ----- Group run 2
 ----- Group run 3
 ----- Group run 4
 ----- Group run 5
 ----- Group run 6
 ----- Group run 7
 ----- Group run 8
 ----- Group run 9
Average Reward: 0.5876
Minibatch 1 Loss: 0.0210, Clipped fraction: 0.2708602304512443
Minibatch 2 Loss: 0.0584, Clipped fraction: 0.3180282464997185
Minibatch 3 Loss: 0.0144, Clipped fraction: 0.3317715575366478
Minibatch 4 Loss: 0.0083, Clipped fraction: 0.33165248192286517
Minibatch 5 Loss: -0.0095, Clipped fraction: 0.32682439187940815
Minibatch 6 Loss: -0.0343, Clipped fraction: 0.32993155472564517
Minibatch 7 Loss: -0.0425, Clipped fraction: 0.3449131763291883
Minibatch 8 Loss: -0.0271, Clipped fraction: 0.3465404079521351
Minibatch 9 Loss: 0.0272, Clipped fraction: 0.3524486882512367
Minibatch 10 Loss: 0.0051, Clipped fraction: 0.3565798785879998
Epoch number 2
 - Trial number 0
 ----- Group run 0
 ----- Group run 1
 ----- Group run 2
 ----- Group run 3
 ----- Group run 4
 ----- Group run 5
 ----- Group run 6
 -